# E - Generalization to other load cases (Cluster 8)

**Reviewer comments addressed**

* **R1.7 / R3.5** - the paper implies the model "generalizes", but it is a
  single geometry / material with two load cases. A standard PINN must be
  **retrained** for new configurations.

We provide the honest, useful version:

1. an **FEM sweep** over voltages `{50, 100, 200} V` and forces `{0.5, 1, 2} N`
   (cheap, exact references);
2. because the problem is **linear**, the trained PINN's prediction scales with
   the load - we show the (load-magnitude) prediction matches FEM at each level;
3. an **optional retraining** cell at a new load (the strategy's "Option A").

The takeaway (Option B in the strategy): reword the generalization claim - the
network handles *load magnitude* by linearity, but genuine generalization to new
geometries/materials would require a parametric or operator-learning approach.

In [ ]:
# === Environment setup (robust: local / Colab native / VSCode-Colab) ===
# Run this cell FIRST. It makes `import pinn_piezo` work regardless of where
# the kernel is, and fails loudly with instructions if it can't.
import os, sys, subprocess
REPO_URL = 'https://github.com/Daniel14gonc/PINNs_piezoelectricity.git'
REPO_DIR = 'PINNs_piezoelectricity'

def _have():
    try:
        import pinn_piezo  # noqa: F401
        return True
    except Exception:
        return False

# 1) Already installed? (e.g. `pip install -e .` locally)
ok = _have()

# 2) Are we *inside* a local checkout? Walk up for src/pinn_piezo.
if not ok:
    d = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(d, 'src', 'pinn_piezo')):
            os.chdir(d); sys.path.insert(0, os.path.join(d, 'src')); break
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    ok = _have()

# 3) Fresh remote runtime (Colab / VSCode-Colab): clone + install.
if not ok:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', '.'], check=True)
    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
    ok = _have()

# scikit-fem is only needed by the FEM cells; install if missing.
try:
    import skfem  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'scikit-fem'], check=True)

# Verify the *new revision modules* are present (i.e. the repo was pushed).
import importlib
missing = [m for m in ('pinn_piezo.fem', 'pinn_piezo.metrics',
                       'pinn_piezo.indirect.standard')
           if importlib.util.find_spec(m) is None]
assert not missing, (
    'These revision modules are missing from the installed package: '
    + ', '.join(missing) + '. Push them to GitHub (git add/commit/push) so the '
    'clone above includes them, then re-run this cell.')

import torch
print('pinn_piezo :', __import__('pinn_piezo').__file__)
print('cwd        :', os.getcwd())
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Where this notebook writes its tables / figures.
from pathlib import Path
OUT = Path('outputs/revision'); OUT.mkdir(parents=True, exist_ok=True)
print('Artefacts ->', OUT.resolve())

## 1. FEM sweep over voltages and forces (exact references)

In [ ]:
import numpy as np, pandas as pd, torch
from pinn_piezo import fem, metrics
from pinn_piezo.config import WIDTH, HEIGHT

rows = []
for V in (50.0, 100.0, 200.0):
    r = fem.solve_piezo('indirect', nx=300, ny=10, voltage=V, poling_sign=-1.0)
    tip = np.abs(r.points[:,0]-WIDTH) < 1e-9
    rows.append({'case':'indirect','load':f'{V:g} V','tip_v_m': r.v[tip].mean(),
                 'phi_max_V': r.phi.max()})
for F in (0.5, 1.0, 2.0):
    r = fem.solve_piezo('direct', nx=300, ny=10, force=F)
    tip = np.abs(r.points[:,0]-WIDTH) < 1e-9
    rows.append({'case':'direct','load':f'{F:g} N','tip_v_m': r.v[tip].mean(),
                 'phi_max_V': np.abs(r.phi).max()})
sweep = pd.DataFrame(rows); sweep.to_csv(OUT / 'fem_load_sweep.csv', index=False); sweep

## 2. Linearity: PINN (trained at reference load) vs FEM at each level

In [ ]:
# Indirect: PINN trained at 100 V; scale by V/100 and compare to FEM at V.
torch.set_default_dtype(torch.float64)
from pinn_piezo.indirect import model as imodel
from pinn_piezo.indirect.train import tensorize
from pinn_piezo.config import MODELS_DIR

paper = imodel.build_default_model(device=DEVICE, model_type='pyramid')
paper.load_state_dict(torch.load(MODELS_DIR/'indirect'/'model_PINN_indirect_paper_3.pt',
                                 map_location=DEVICE)); paper.eval()

rows = []
for V in (50.0, 100.0, 200.0):
    r = fem.solve_piezo('indirect', nx=300, ny=10, voltage=V, poling_sign=-1.0)
    XY = r.points
    p = paper(tensorize(XY, DEVICE)).detach().cpu().numpy()
    scale = V / 100.0   # linear scaling of the 100 V network
    pred = {'u': p[:,0]*scale, 'v': p[:,1]*scale, 'phi': p[:,2]*scale}
    ref = {'u': r.u, 'v': r.v, 'phi': r.phi}
    mt = metrics.metrics_table(pred, ref)
    rows.append({'V': V, 'L2_u': mt.loc['u','rel_L2'], 'L2_v': mt.loc['v','rel_L2'],
                 'L2_phi': mt.loc['phi','rel_L2']})
lin = pd.DataFrame(rows).set_index('V')
lin.to_csv(OUT / 'indirect_linearity_vs_fem.csv'); lin

The relative L2 should be ~constant across voltages: the error comes from the
network, not the load level - i.e. the PINN handles load magnitude by linearity,
**not** by learned generalization. State this honestly.

## 3. (Optional) Retrain at a new load - the strategy's 'Option A'

In [ ]:
# Indirect at a new voltage: set config.VOLTAGE, rebuild & retrain, score vs FEM.
RUN_RETRAIN = False  # set True for the full (slower) experiment
if RUN_RETRAIN:
    from pinn_piezo import config
    from pinn_piezo.indirect import train as itrain
    config.VOLTAGE = 200.0   # phi_constraint reads this at call time
    arrays = itrain.load_dataset('data', suffix='_m1', fraction=1.0)
    tensors = itrain.to_device(arrays, DEVICE, dtype=torch.float64)
    np.random.seed(0); torch.manual_seed(0)
    m = imodel.build_default_model(device=DEVICE, model_type='pyramid')
    itrain.train(m, tensors, epochs_adam=1000, epochs_lbfgs=200)
    r = fem.solve_piezo('indirect', nx=300, ny=10, voltage=200.0, poling_sign=-1.0)
    p = m(tensorize(r.points, DEVICE)).detach().cpu().numpy()
    print(metrics.metrics_table({'u':p[:,0],'v':p[:,1],'phi':p[:,2]},
                                {'u':r.u,'v':r.v,'phi':r.phi}))
    config.VOLTAGE = 100.0   # restore

# Direct at a new force: set the module-level applied force then retrain.
if RUN_RETRAIN:
    from pinn_piezo.direct import losses as dlosses, model as dmodel, train as dtrain
    torch.set_default_dtype(torch.float32)
    dlosses.APPLIED_FORCE_Y = 2.0
    arrays = dtrain.load_dataset('data', suffix='_m1_d', fraction=0.75)
    tensors = dtrain.to_device(arrays, DEVICE, dtype=torch.float32)
    md_ = dmodel.build_default_model(device=DEVICE)
    dtrain.train(md_, tensors, epochs_adam=3000, epochs_lbfgs=0)
    dlosses.APPLIED_FORCE_Y = 0.1  # restore

---
### Rebuttal snippet (Cluster 8)
> *We clarified the generalization claim. The framework is linear, so a network
> trained at one load reproduces other load magnitudes by scaling (Table …,
> relative L2 is load-independent), and we verified this against FEM at
> 50/100/200 V and 0.5/1/2 N. We note that genuine generalization across
> geometries or materials would require a parametric or operator-learning
> extension (DeepONet/FNO), which we now list as future work rather than a
> demonstrated capability.*